In [ ]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import torch.nn as nn
import numpy as np
import shutil
from torchvision.transforms import ToPILImage
from metrics import compute_mae
from plot import plots
import math
import torchvision.transforms as T
import random

import os
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F

import time
import psutil


import matplotlib
#matplotlib.use("TkAgg") 
import matplotlib.pyplot as plt
 
import random
from torch.nn import functional as F
from torchvision import transforms as T

In [ ]:
to_pil = ToPILImage()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
def set_seed(seed: int = 42):
    """
    Fixes all random seeds to ensure reproducible training results.
    """
    # 1. Python core randomness
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. NumPy randomness
    np.random.seed(seed)
    
    # 3. PyTorch CPU randomness
    torch.manual_seed(seed)
    
    # 4. PyTorch GPU randomness (if applicable)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # For multi-GPU setups
        
        # 5. CUDA Convolution Algorithms Stability
        # Forces CUDA to always select the exact same deterministic algorithms
        torch.backends.cudnn.deterministic = True
        # Disables auto-tuning benchmarking which can introduce slight numerical variations
        torch.backends.cudnn.benchmark = False
        
    print(f"✅ Random seed set to: {seed} (Deterministic mode enabled)")



In [ ]:
# =========================
# 0️⃣️ --- DrivingDataset --- 
# =========================
class MultiTaskDataset(Dataset):
    """
    Expected structure:
    root/
      control_images/
      seg_images/
      seg_masks/
      angle/
      velocity/
    """

    def __init__(self, root, max_, min_, max_a_, min_a_, ctl_transforms=None, img_transforms=None, mask_transforms=None):
        self.root = root

        self.ctl_imgs  = sorted(os.listdir(root + "/control/images"))
        self.seg_imgs  = sorted(os.listdir(root + "/weed_crop/images"))
        self.masks     = sorted(os.listdir(root + "/weed_crop/masks"))
        self.angles    = sorted(os.listdir(root + "/control/angle"))
        self.vels      = sorted(os.listdir(root + "/control/velocity"))

        self.img_transforms   = img_transforms
        self.mask_transforms  = mask_transforms
        self.ctl_transforms   = ctl_transforms
        self.max_             = max_
        self.min_             = min_
        self.max_a_           = max_a_
        self.min_a_           = min_a_

        self.trans = T.ToTensor()

    def __len__(self):
       if self.seg_imgs is not None:
          assert len(self.seg_imgs)==len(self.masks), "The number of images is not the same as the number of masks:  "
       else:
          return 0
       return len(self.seg_imgs)

    def __getitem__(self, idx): 
        img_src   = os.path.join(self.root, "weed_crop/images", self.seg_imgs[idx])
        seg_img       = Image.open(img_src)
        
        if self.img_transforms is not None:
           seg_img = self.img_transforms(seg_img)
        else:  
           seg_img = self.trans(seg_img) 
        
        mask_src  = os.path.join(self.root, "weed_crop/masks", self.masks[idx])
        mask      = Image.open(mask_src)
        
        if self.mask_transforms is not None:
           mask = self.mask_transforms(mask)
        else:  
           mask = self.trans(mask) 
           
        mask_max = mask.max().item()
        if mask_max > 0:
           mask /= mask_max

        img_ctl_src   = os.path.join(self.root, "control/images", self.ctl_imgs[idx])
        ctl_img       = Image.open(img_ctl_src)
        
        if self.ctl_transforms is not None:
           ctl_img = self.ctl_transforms(ctl_img)
        else:  
           ctl_img = self.trans(ctl_img) 
        
        angle_src = os.path.join(self.root, "control/angle",    self.angles[idx])
        with open(angle_src, "r") as f:
            angle_cont = f.read().strip()
            angle = (float(angle_cont) - self.min_a_) / (self.max_a_ - self.min_a_)
               
        
        vel_src   = os.path.join(self.root, "control/velocity", self.vels[idx])
        with open(vel_src, "r") as f:
            vel_m = f.read().strip()
            #vel_cont, vel_m =   vel_c.split()
            normalized_vel = (float(vel_m)-self.min_)/(self.max_-self.min_)
            velocity=float(normalized_vel)
               
        return seg_img, mask, ctl_img, angle, velocity   

In [ ]:
#====================================
#1️⃣️ --get min max velocity values ---
#====================================
def get_min_max_values(root):
    v_ = []
    g_ = []
    
    vel_list = sorted(os.listdir(root + "/control/velocity"))
    ang_list = sorted(os.listdir(root + "/control/angle"))
   
    for i, f_ang in enumerate(ang_list):
       ang_src = os.path.join(root + "/control/angle/" + f_ang)
        
       with open(ang_src, "r") as g:
          ang_cont= g.read().strip() 
          ###print(f"i:{i} | angle:= {float(ang_cont):.3f}")
          if ang_cont == '':
             #print(f"Empty file at index {i}: {ang_src}")
             ang_cont = "90"
          g_.append(float(ang_cont))
    
    
    for i, f_vel in enumerate(vel_list):
       vel_src = os.path.join(root + "/control/velocity/" + f_vel)
       
       with open(vel_src, "r") as f:
          velocity_cont= f.read().strip()
           
          if len(velocity_cont)== 0:
              velocity_cont = "0"
             
          v_.append(float(velocity_cont))
                 
    max_vel = max(v_)
    min_vel = min(v_)
    
    
    max_ang = max(g_)
    min_ang = min(g_)
    
    
    return max_vel, min_vel, max_ang, min_ang

In [ ]:
def plot_mini_batch(imgs, masks,BATCH_SIZE):
    plt.figure(figsize=(20,10))
    for i in range(BATCH_SIZE):
        plt.subplot(4, 8, i+1)
        img=imgs[i,...].permute(1,2,0).numpy()
        mask = masks[i, ...].permute(1,2,0).numpy()
        plt.imshow(img)
        plt.imshow(mask, alpha=0.7)
        
        plt.axis('Off')
    plt.tight_layout()
    plt.show()

In [ ]:
class Conv_3_k(nn.Module):
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.conv1 = nn.Conv2d(channels_in, channels_out, kernel_size=3, stride=1, padding=1)
    def forward(self, x):
        return self.conv1(x)

In [ ]:
class Double_Conv(nn.Module):
    '''
    Double convolution block for U-Net
    '''
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.double_conv = nn.Sequential(
                           Conv_3_k(channels_in, channels_out),
                           nn.BatchNorm2d(channels_out),
                           nn.ReLU(),
            
                           Conv_3_k(channels_out, channels_out),
                           nn.BatchNorm2d(channels_out),
                           nn.ReLU(),
                            )
    def forward(self, x):
        return self.double_conv(x)
    
class Down_Conv(nn.Module):
    '''
    Down convolution part
    '''
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.encoder = nn.Sequential(
                        nn.MaxPool2d(2,2),
                        Double_Conv(channels_in, channels_out)
                        )
    def forward(self, x):
        return self.encoder(x)
    
class Up_Conv(nn.Module):
    '''
    Up convolution part
    '''
    def __init__(self,channels_in, channels_out):
        super().__init__()
        self.upsample_layer = nn.Sequential(
                        nn.Upsample(scale_factor=2, mode='bicubic'),
                        nn.Conv2d(channels_in, channels_in//2, kernel_size=1, stride=1)
                        )
        self.decoder = Double_Conv(channels_in, channels_out)
    
    def forward(self, x1, x2):
        '''
        x1 - upsampled volume
        x2 - volume from down sample to concatenate
        '''
        x1 = self.upsample_layer(x1)
        x = torch.cat([x2, x1],dim=1)
        return self.decoder(x)
    
class UNET(nn.Module):
    '''
    UNET model
    '''
    def __init__(self, channels_in, channels, num_classes):
        super().__init__()
        self.first_conv = Double_Conv(channels_in, channels) #64, 224, 224
        self.down_conv1 = Down_Conv(channels, 2*channels) # 128, 112, 112
        self.down_conv2 = Down_Conv(2*channels, 4*channels) # 256, 56, 56
        self.down_conv3 = Down_Conv(4*channels, 8*channels) # 512, 28, 28
        
        self.middle_conv = Down_Conv(8*channels, 16*channels) # 1024, 14, 14 
        
        self.up_conv1 = Up_Conv(16*channels, 8*channels)
        self.up_conv2 = Up_Conv(8*channels, 4*channels)
        self.up_conv3 = Up_Conv(4*channels, 2*channels)
        self.up_conv4 = Up_Conv(2*channels, channels)
        
        self.last_conv = nn.Conv2d(channels, num_classes, kernel_size=1, stride=1)
        
    def forward(self, x):
        x1 = self.first_conv(x)
        x2 = self.down_conv1(x1)
        x3 = self.down_conv2(x2)
        x4 = self.down_conv3(x3)
        
        x5 = self.middle_conv(x4)
        
        u1 = self.up_conv1(x5, x4)
        u2 = self.up_conv2(u1, x3)
        u3 = self.up_conv3(u2, x2)
        u4 = self.up_conv4(u3, x1)
        
        return self.last_conv(u4)

In [ ]:
class CNN_FC(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3,  24,  kernel_size=3, stride=1, padding=1)
        self.bn1   = nn.BatchNorm2d(24) 
        
        self.conv2 = nn.Conv2d(24, 32,  kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        
        self.conv3 = nn.Conv2d(32, 48,  kernel_size=3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm2d(48)
        
        self.conv4 = nn.Conv2d(48, 64,  kernel_size=3, stride=1, padding=1)
        self.bn4   = nn.BatchNorm2d(64)
        
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn5   = nn.BatchNorm2d(128)
        
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.dropout = nn.Dropout(p=0.01)

        self.fc1   = nn.Linear(128*8*8, 100)
        self.fc2   = nn.Linear(100, 50)
        self.fc3   = nn.Linear(50, 10)
        self.fc4   = nn.Linear(10, 2)  # steering + velocity
        
        self.flatten = nn.Flatten()
        
    def forward(self, x):
       x = F.relu(self.bn1(self.conv1(x)))
       x = self.max_pool(x)
       
       x = F.relu(self.bn2(self.conv2(x)))
       x = self.max_pool(x)
       
       x = F.relu(self.bn3(self.conv3(x)))
       x = self.max_pool(x)
       
       x = F.relu(self.bn4(self.conv4(x)))
       x = self.max_pool(x)
       
       x = F.relu(self.bn5(self.conv5(x)))
       x = self.max_pool(x)
       
      
       
       x = self.flatten(x)
       
       x = F.relu(self.fc1(x))
      
       x = F.relu(self.fc2(x))
       
       x = F.relu(self.fc3(x))
      
       x = self.fc4(x)
       
       return x

In [ ]:
def accuracy(model, loader):
    correct      = 0
    intersection = 0
    denom        = 0
    union        = 0
    total        = 0
    cost         = 0.

    # For mIoU and mPA
    tp = 0
    fp = 0
    fn = 0
    tn = 0

    model = model.to(device=device)

    with torch.no_grad():
        for x, y, ctl_img, angle, velocity in loader:

            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long).squeeze(1)

            scores = model(x)

            cost += (F.cross_entropy(scores, y)).item()

            preds = torch.argmax(scores, dim=1)

            # standard accuracy
            correct += (preds == y).sum()
            total += torch.numel(preds)

            # dice coefficient
            intersection += (preds * y).sum()
            denom += (preds + y).sum()

            # intersection over union
            union += (preds + y - preds * y).sum()

            # confusion matrix
            tp += ((preds == 1) & (y == 1)).sum().item()
            fp += ((preds == 1) & (y == 0)).sum().item()
            fn += ((preds == 0) & (y == 1)).sum().item()
            tn += ((preds == 0) & (y == 0)).sum().item()

        dice = 2 * intersection / (denom + 1e-8)
        iou  = intersection / (union + 1e-8)

        # convert tensors to scalars
        dice = dice.item()
        iou  = iou.item()

        # mIoU
        iou_weed = tp / (tp + fp + fn + 1e-8)
        iou_bg   = tn / (tn + fp + fn + 1e-8)
        miou     = (iou_weed + iou_bg) / 2.0

        # mPA
        pa_weed = tp / (tp + fn + 1e-8)
        pa_bg   = tn / (tn + fp + 1e-8)
        mPA     = (pa_weed + pa_bg) / 2.0

        return cost/len(loader), float(correct)/total, dice, iou, miou, mPA

In [ ]:
#================
# ---- TRAIN ----
#================
def train(model_seg,model_ctl, train_loader, val_loader, test_loader, optimiser_unet, optimizer_ctl, criterion, device,
          epochs, store_every, max_ang, min_ang, max_vel, min_vel, scheduler = None): 
    model_seg = model_seg.to(device=device)
    model_ctl = model_ctl.to(device=device)
    train_losses    = []
    val_losses      = []
    test_losses     = []

    train_seg_losses = []
    val_seg_losses   = []
    test_seg_losses  = []

    train_pix_acc_list = []
    val_pix_acc_list = []
    test_pix_acc_list = []

    train_dice_list = []
    val_dice_list = []
    test_dice_list = []

    train_iou_list = []
    val_iou_list = []
    test_iou_list = []

    train_miou_list = []
    val_miou_list = []
    test_miou_list = []

    train_mPA_list = []
    val_mPA_list = []
    test_mPA_list = []

    best_val_loss = float("inf")
    best_val_dice = float("-inf")

    #### MEDICIÓN: Inicializar acumuladores para el bucle conjunto ###
    total_inference_time   = 0.0
    pick_vram_loop         = 0.0
    total_images_processed = 0
    val_loader_acc         = 0
    
    for epoch in range(epochs):   
       train_loss        = 0.0
       train_total       = 0
       train_correct_num = 0
       train_cost_acum   = 0.
       model_seg.train()
       model_ctl.train()
       #=========================
       # --- Train UNET loader --
       #========================= 
       for nb, (seg_img, mask, ctl_img, ang, vel)  in enumerate(train_loader, start=1):
           seg_img = seg_img.to(device=device, dtype=torch.float32)
           mask = mask.to(device=device, dtype=torch.long).squeeze(1)
           ctl_img = ctl_img.to(device=device, dtype=torch.float32)
           ang = ang.to(device=device, dtype=torch.float32)
           vel = vel.to(device=device, dtype=torch.float32)
           ctrl_target = torch.stack((ang, vel), dim=1).to(device=device)
           
           scores = model_seg(seg_img)

           weights = torch.tensor([1.0, 2.0]).to(device)
           cost = F.cross_entropy(input=scores, target=mask, weight=weights)

            
           optimiser_unet.zero_grad()
           cost.backward()
           optimiser_unet.step()
                
           train_predictions = torch.argmax(scores, dim=1)
           
           train_correct_num += (train_predictions == mask).sum().item()
           ##train_total += torch.numel(train_predictions)
           train_total += mask.numel()
           train_cost_acum += cost.item()
           if nb%store_every == 0:
               val_cost, val_acc, dice, iou, miou, mPA = accuracy(model_seg, train_loader)
               train_acc = float(train_correct_num)/train_total
               train_cost_every = float(train_cost_acum)/nb
               
               print(f'epoch: {epoch+1}, nb: {nb}, train cost: {train_cost_every:.4f}, val cost: {val_cost:.4f},'
                     f'train acc: {train_acc:.4f}, val acc: {val_acc:.4f},'
                     f'dice: {dice:.4f}, iou: {iou:.4f}')

       

       
           #============================
           # --- Train CNN-FC loader ---
           #============================ 
           preds = model_ctl(ctl_img)
           loss = criterion(preds, ctrl_target)
           optimizer_ctl.zero_grad()
           loss.backward()
           optimizer_ctl.step()
            
           train_loss += loss.item()
       train_loss /= len(train_loader)
       train_losses.append(train_loss)
        
       #=======================
       # --- VAL the CNN-FC ---
       #======================= 
       model_ctl.eval()
       model_seg.eval()
       val_loss = 0.0
       test_loss = 0.0 

       
        
       with torch.no_grad():
          for nb, (seg_img, mask, ctl_img, ang, vel)  in enumerate(val_loader, start=1):
             seg_img = seg_img.to(device=device, dtype=torch.float32)
             mask = mask.to(device=device, dtype=torch.long).squeeze(1)
             ctl_img = ctl_img.to(device=device, dtype=torch.float32)
             ang = ang.to(device=device, dtype=torch.float32)
             vel = vel.to(device=device, dtype=torch.float32)
             ctrl_target = torch.stack((ang, vel), dim=1).to(device=device)

             # ### MEDICIÓN: Sincronizar y tomar tiempo inicial del lote ###
             if torch.cuda.is_available(): torch.cuda.synchronize()
             t_start = time.perf_counter() 

             # --- INFERENCIA CONJUNTA (Ambos modelos ejecutan en el mismo ciclo) ---
             preds = model_ctl(ctl_img)
             scores = model_seg(seg_img) 

             # ### MEDICIÓN: Sincronizar y calcular delta de tiempo ###
             if torch.cuda.is_available(): torch.cuda.synchronize()
             t_end = time.perf_counter()
         
             total_inference_time += (t_end - t_start)
             total_images_processed += seg_img.size(0) # Acumular tamaño de lote (batch_size) 

             val_loader_acc += 1
              
             # Cálculo de métricas normales del bucle
             loss = criterion(preds, ctrl_target)
             val_loss += loss.item()
             cost = F.cross_entropy(input=scores, target=mask)

             # ### MEDICIÓN: Monitorear memoria en cada paso ###
             if torch.cuda.is_available():
                vram_actual = torch.cuda.max_memory_allocated(device=device) / (1024**2) # En MB
                if vram_actual > pick_vram_loop:
                   pick_vram_loop = vram_actual

          # --- FIN DEL BUCLE DE VALIDACIÓN: CÁLCULO DE MÉTODOS REPORTABLES ---
          avg_inference_time_batch = (total_inference_time / val_loader_acc) * 1000 # En milisegundos por lote
          fps_system = total_images_processed / total_inference_time # Cuántas imágenes procesó por segundo

         
      
          # Consumo de memoria RAM del Sistema (Host CPU)
          system_process = psutil.Process(os.getpid())
          consumption_mb_ram = system_process.memory_info().rss / (1024**2)

          print(f"\n📈 --- CO-EXECUTION REPORT (EPOCH {epoch+1}) ---")
          print(f"Nunber of batches {nb} and seg_img.size(0) {seg_img.size(0)}" 
                f"  and total_images_processed {total_images_processed}, val_loader: {len(val_loader)}")
          print(f"├─ Average Inference Time (Batch): {avg_inference_time_batch:.2f} ms")
          print(f"├─ Combined Throughput:            {fps_system:.2f} FPS")
          print(f"├─ RAM Consumption (Process):      {consumption_mb_ram:.2f} MB")
          if torch.cuda.is_available():
             print(f"└─ Peak VRAM Usage (GPU):          {pick_vram_loop:.2f} MB")
          print("----------------------------------------------------------\n")


          # Resetear estadísticas de memoria CUDA para la siguiente época
          if torch.cuda.is_available():
              torch.cuda.reset_peak_memory_stats(device=device)

      # =======================
      # --- BUCLE DE TEST ----
      # =======================
        
          for nb, (seg_img, mask, ctl_img, ang, vel)  in enumerate(test_loader, start=1):
             seg_img = seg_img.to(device=device, dtype=torch.float32)
             mask = mask.to(device=device, dtype=torch.long).squeeze(1)
             ctl_img = ctl_img.to(device=device, dtype=torch.float32)
             ang = ang.to(device=device, dtype=torch.float32)
             vel = vel.to(device=device, dtype=torch.float32)
             ctrl_target = torch.stack((ang, vel), dim=1).to(device=device)
              
             preds = model_ctl(ctl_img)
             loss = criterion(preds, ctrl_target)
             test_loss += loss.item()    


      
       #================================
       # --- Save best control model ---
       #================================ 
       if val_loss < best_val_loss:
          best_val_loss = val_loss
          torch.save(model_ctl.state_dict(), "best_control_model.pt")
          print("🦺 Saved BEST control model")


       #================================
       # --- Compute Control metrics ---
       #================================ 
       #==================
       # --- TRAIN MAE ---
       #==================
       train_ang_err, train_vel_err, train_rmse_vel, train_rmse_ang   = compute_mae(model_ctl, train_loader, 
                                                                                    device, max_ang, min_ang, 
                                                                                    max_vel, min_vel)

       #================
       # --- VAL MAE ---
       #================
       val_ang_err, val_vel_err, val_rmse_vel, val_rmse_ang   = compute_mae(model_ctl, val_loader, 
                                                                            device, max_ang, min_ang, 
                                                                            max_vel, min_vel)

      

       #=====================
       # --- TEST MAE ---
       #=====================
       test_ang_err, test_vel_err, test_rmse_vel, test_rmse_ang   = compute_mae(model_ctl, test_loader, 
                                                                                device, max_ang, 
                                                                                min_ang, max_vel, min_vel)
        
       #===============================
       # ---- Compute UNET metrics ----
       #=============================== 
       train_cost, train_pix_acc, train_dice, train_iou, train_miou, train_mPA  = accuracy(model_seg, train_loader)
       print(f'train_loader: ,'
             f'epoch: {epoch+1}, nb: {nb}, train cost: {train_cost_every:.4f}, val cost: {train_cost:.4f},'
             f'train pix acc: {train_pix_acc:.4f},'
             f'dice: {dice:.4f}, iou: {train_iou:.4f},'
             f'train_miou: {train_miou:.4f}, train_mPA: {train_mPA:.4f}')

       val_cost, val_pix_acc, val_dice, val_iou, val_miou, val_mPA = accuracy(model_seg, val_loader)
       print(f'val_loader:   ,'
             f'epoch: {epoch+1}, nb: {nb}, train cost: {train_cost_every:.4f}, val cost: {val_cost:.4f},'
             f'val pix acc: {val_pix_acc:.4f},'
             f'dice: {dice:.4f}, iou: {iou:.4f},'
             f'val_miou: {val_miou:.4f}, val_mPA: {val_mPA:.4f}')

       test_cost, test_pix_acc, test_dice, test_iou, test_miou, test_mPA = accuracy(model_seg, test_loader)
       print(f'test_loader:  ,'
             f'epoch: {epoch+1}, nb: {nb}, train cost: {train_cost_every:.4f}, val cost: {test_cost:.4f},'
             f'test pix acc: {test_pix_acc:.4f},'
             f'dice: {dice:.4f}, iou: {iou:.4f},'
             f'test_miou: {test_miou:.4f}, test_mPA: {test_mPA:.4f}')

       print("\n")
       #===================
       # calculate segmentation loss
       #=================== 
       train_seg_losses.append(train_cost)
       val_seg_losses.append(val_cost)
       test_seg_losses.append(test_cost)
       #=============================
       # --- Save best UNet model ---
       #=============================
       if val_dice > best_val_dice:
          best_val_dice = val_dice
          torch.save(model_seg.state_dict(), "best_unet.pt")
          print("🧠 Saved BEST UNet model")       


       train_pix_acc_list.append(train_pix_acc)
       val_pix_acc_list.append(val_pix_acc)
       test_pix_acc_list.append(test_pix_acc)

       train_dice_list.append(train_dice)
       val_dice_list.append(val_dice)
       test_dice_list.append(test_dice)

       train_iou_list.append(train_iou)
       val_iou_list.append(val_iou)
       test_iou_list.append(test_iou)

       train_miou_list.append(train_miou)
       val_miou_list.append(val_miou)
       test_miou_list.append(test_miou)

       train_mPA_list.append(train_mPA)
       val_mPA_list.append(val_mPA)
       test_mPA_list.append(test_mPA) 

        
         
       
       val_loss   /= len(val_loader)
       val_losses.append(val_loss)

       test_loss   /= len(test_loader)
       test_losses.append(test_loss) 
        
       #EarlyStopping()
              
       
       print(f"Train Loader: {len(train_loader):.6f} | Val Loader: {len(val_loader):.6f} | Test Loader: {len(test_loader):.6f}")
       print(f"Epoch {epoch+1}/{epochs} | Train control loss: {train_loss:.6f} | Val control loss: {val_loss:.6f} | Test contro loss: {test_loss:.6f}")
       print(f"Train seg loss: {train_loss:.6f} | Val seg loss: {val_loss:.6f} | Test seg loss: {test_loss:.6f}")
       print("\n")
    


    return {
               "train_dice":     train_dice_list,
               "val_dice":       val_dice_list,
               "test_dice":      test_dice_list,
               "train_iou":      train_iou_list,
               "val_iou":        val_iou_list,
               "test_iou":       test_iou_list,
               "train_miou":      train_miou_list,
               "val_miou":        val_miou_list,
               "test_miou":       test_miou_list,
               "train_mPA":      train_mPA_list,
               "val_mPA":        val_mPA_list,
               "test_mPA":       test_mPA_list,
               "train_pix_acc":  train_pix_acc_list,
               "val_pix_acc":    val_pix_acc_list,
               "test_pix_acc":   test_pix_acc_list,
               "train_ang_err":  train_ang_err, 
               "train_vel_err":  train_vel_err,
               "train_rmse_vel": train_rmse_vel,
               "train_rmse_ang": train_rmse_ang,
               "val_ang_err":    val_ang_err,
               "val_vel_err":    val_vel_err,
               "val_rmse_vel":   val_rmse_vel,
               "val_rmse_ang":   val_rmse_ang,
               "test_ang_err":   test_ang_err,
               "test_vel_err":   test_vel_err, 
               "test_rmse_vel":  test_rmse_vel, 
               "test_rmse_ang":  test_rmse_ang,
               "train_losses":   train_losses,
               "val_losses":     val_losses,
               "test_losses":    test_losses,
               "train_seg_losses": train_seg_losses,
               "val_seg_losses":   val_seg_losses,
               "test_seg_losses":  test_seg_losses
          }
       

In [ ]:
def main():
    # --- CALL IT IMMEDIATELY ---
    set_seed(42)  # 42 is the industry standard choice, but any integer works!
    root = "/path/to/your/root/working/directory"
    batch_size    = 4 #####16
    epochs        = 3 #####100	
    #best_val_loss = float("inf")
    store_every   = 2 

    t_ctrl = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])

    t_seg = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor()
    ])
    #=============================================
    # --- Get min and max angles and velocities --
    #=============================================
    max_vel, min_vel, max_ang, min_ang = get_min_max_values(root)
    print("min_vel",  min_vel)
    print("max_vel",  max_vel)
    
    print("min_ang",  min_ang)
    print("max_ang",  max_ang)



    #====================
    # --- Get dataset ---
    #====================
    dataset = MultiTaskDataset(root, max_vel, min_vel, max_ang, min_ang, ctl_transforms=t_ctrl, img_transforms=t_seg, mask_transforms=t_seg)
    print("len_dataset: ",  len(dataset))
    
    train_size = int(0.7 * len(dataset))
    val_size   = int(0.15 * len(dataset))
    test_size  = len(dataset) - train_size - val_size

    
    # ==========================================================
    # MANUALLY SEED THE SPLIT generator
    # ==========================================================
    # PyTorch random_split requires an explicit generator to guarantee 
    # that the exact same images land in train/val/test on every run.
    g = torch.Generator().manual_seed(42)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size], generator=g
    )

    print(f"train_dataset: {len(train_dataset)} | val_dataset: {len(val_dataset)} | test_dataset:  {len(test_dataset)}")
   
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=True, drop_last=False)
    test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=True, drop_last=False)

    print(f"train_loader: {len(train_loader)} | val_loader: {len(val_loader)} | test_loader:  {len(test_loader)}")
   

    for i, (seg_img, mask, ctl_img, angle, velocity) in enumerate(train_loader):
       print(
             f"i:{i} | seg: {seg_img.shape} | mask: {mask.shape} | "
             f"ctl: {ctl_img.shape} | agl: {angle.shape} | "
             f"vel: {velocity.shape}"
            )
       if i==1: break
           
    plot_mini_batch(seg_img, mask, 4)

    #=====================
    # --- Train Models ---
    #=====================
    model_seg = UNET(channels_in=3, channels=4, num_classes=2)
    model_ctl = CNN_FC()

    
    criterion = nn.SmoothL1Loss()
    optimizer_ctl = optim.Adam(model_ctl.parameters(), lr=1e-4)
    optimiser_unet = torch.optim.SGD(model_seg.parameters(),
                                 lr=1e-4, momentum=0.95, #lr=0.06
                                 weight_decay=1e-4)
   
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimiser_unet, 
                                                max_lr = 1e-1,
                                                steps_per_epoch=len(train_loader),
                                                epochs=epochs, pct_start=0.43, div_factor=10, final_div_factor=1000,
                                                three_phase=True)
    
    metrics =  train(model_seg, model_ctl, train_loader, val_loader, 
               test_loader, optimiser_unet, optimizer_ctl, 
               criterion, device, epochs, store_every, max_ang, 
               min_ang, max_vel, min_vel,scheduler)       

    plots(metrics)
    
    imgs_val, masks_val, ctl_img, angle, velocity= next(iter(val_loader))
    imgs_val = imgs_val.to(device, dtype=torch.float32)
    model = model_seg.to(device)
    with torch.no_grad():
       scores = model(imgs_val)
       preds = torch.argmax(scores, dim=1).float()

    imgs_val = imgs_val.cpu()
    preds = preds.cpu()
    print(preds.shape)
    plot_mini_batch(imgs_val, preds.unsqueeze(1), 4)

    dir_src = "/path/to/your/Figures/directory"
    for i in range(imgs_val.size(0)):
       # Save input image
       img = imgs_val[i].cpu()
       img_pil = to_pil(img)
       img_pil.save(os.path.join(dir_src,f"img_{i}.png"))

       # Save prediction (mask)
       pred = preds[i].unsqueeze(0)  # add channel
       pred_pil = to_pil(pred)
       pred_pil.save(os.path.join(dir_src,f"pred_{i}.png"))

    

In [ ]:
if __name__ == "__main__":
    main()